In [1]:
"""
LNG Feedgas Dashboard - Auto-Update & Deploy to GitHub
========================================================
Run this script to:
1. Fetch latest feedgas data from S&P Global API
2. Generate the HTML dashboard
3. Commit and push to GitHub automatically

To auto-run daily: see Task Scheduler instructions at bottom
"""

import requests
import json
import os
import subprocess
from datetime import datetime

# ── CONFIG ────────────────────────────────────────────────────────────────────
# API Credentials
USERNAME = "39a9cfa5-18cf-4f0b-b66c-1634b79e906e"
PASSWORD = "cpLScP8pMDGywJks"

# Local paths
REPO_DIR = r"C:\Users\melaea\LNGFeedgas"  # Your GitHub repo directory
DATA_FILE = os.path.join(REPO_DIR, "lng_feedgas_clean.json")
HTML_FILE = os.path.join(REPO_DIR, "index.html")  # GitHub Pages uses index.html

# Data range
START_DATE = "1/1/2025"
# ─────────────────────────────────────────────────────────────────────────────

FACILITIES = [
    'Sabine Pass', 'Plaquemines LNG', 'Corpus Christi', 'Freeport',
    'Cameron', 'Calcasieu Pass', 'Cove Point', 'Elba Island', 'Golden Pass'
]

COLORS = {
    'Sabine Pass': '#00b4d8',
    'Plaquemines LNG': '#ff6b35',
    'Corpus Christi': '#a78bfa',
    'Freeport': '#2ecc71',
    'Cameron': '#ffd166',
    'Calcasieu Pass': '#f72585',
    'Cove Point': '#4cc9f0',
    'Elba Island': '#fb8500',
    'Golden Pass': '#80ffdb',
}


def fetch_data():
    """Fetch feedgas data from S&P Global API with retry logic."""
    today = datetime.today().strftime("%#m/%#d/%Y") if os.name == 'nt' else datetime.today().strftime("%-m/%-d/%Y")
    print(f"📡 Fetching data from {START_DATE} to {today}...")
    
    max_retries = 3
    timeout = 60  # Increased to 60 seconds
    
    for attempt in range(max_retries):
        try:
            if attempt > 0:
                print(f"   ⟳ Retry attempt {attempt + 1}/{max_retries}...")
            
            r = requests.get(
                "https://api.connect.spglobal.com/cs/v1/pointlogic/lngFlowByFacility",
                params={"startDate": START_DATE, "endDate": today},
                auth=(USERNAME, PASSWORD),
                timeout=timeout
            )
            r.raise_for_status()
            
            feedgas = [rec for rec in r.json().get("Data", []) if rec["type"] == "Feedgas"]
            print(f"   ✓ {len(feedgas)} feedgas records fetched")
            return feedgas
            
        except requests.exceptions.Timeout:
            if attempt < max_retries - 1:
                print(f"   ⚠️  Request timed out after {timeout}s, retrying...")
                continue
            else:
                print(f"   ❌ Request timed out after {max_retries} attempts")
                raise
        except requests.exceptions.RequestException as e:
            if attempt < max_retries - 1:
                print(f"   ⚠️  Error: {e}, retrying...")
                continue
            else:
                raise


def save_data(feedgas):
    """Save raw data to JSON file."""
    with open(DATA_FILE, "w") as f:
        json.dump(feedgas, f, indent=2)
    print(f"   ✓ Data saved to {DATA_FILE}")


def generate_html(feedgas):
    """Generate the full standalone HTML dashboard."""
    compact = [{"d": r["date"], "f": r["facilityname"], "v": r["volume"]} for r in feedgas]
    data_str = json.dumps(compact, separators=(',', ':'))

    dates = sorted(set(r["date"] for r in feedgas))
    date_range = f"{dates[0]}  →  {dates[-1]}" if dates else "No data"
    generated_at = datetime.now().strftime("%B %d, %Y at %I:%M %p")

    colors_js = json.dumps(COLORS)
    facilities_js = json.dumps(FACILITIES)

    html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>LNG Feedgas Dashboard</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.min.js"></script>
<link href="https://fonts.googleapis.com/css2?family=Bebas+Neue&family=IBM+Plex+Mono:wght@400;600&family=IBM+Plex+Sans:wght@300;400;600&display=swap" rel="stylesheet">
<style>
  :root {{
    --bg:#0a0e17; --panel:#111827; --border:#1e2d45;
    --accent:#00d4ff; --text:#e2e8f0; --muted:#64748b;
  }}
  *{{margin:0;padding:0;box-sizing:border-box;}}
  body{{background:var(--bg);color:var(--text);font-family:'IBM Plex Sans',sans-serif;min-height:100vh;
    background-image:radial-gradient(ellipse at 20% 0%,rgba(0,212,255,.06) 0%,transparent 50%),
    radial-gradient(ellipse at 80% 100%,rgba(255,107,53,.05) 0%,transparent 50%);}}
  header{{padding:28px 40px 18px;border-bottom:1px solid var(--border);display:flex;align-items:flex-end;gap:20px;}}
  .logo-mark{{width:4px;height:44px;background:linear-gradient(180deg,#00d4ff,#ff6b35);border-radius:2px;flex-shrink:0;}}
  h1{{font-family:'Bebas Neue',sans-serif;font-size:2.6rem;letter-spacing:.08em;line-height:1;color:#fff;}}
  .subtitle{{font-family:'IBM Plex Mono',monospace;font-size:.68rem;color:var(--accent);letter-spacing:.15em;text-transform:uppercase;margin-bottom:3px;}}
  .date-badge{{font-family:'IBM Plex Mono',monospace;font-size:.68rem;color:var(--muted);margin-left:auto;text-align:right;line-height:1.8;}}
  .updated{{font-size:.6rem;color:var(--muted);opacity:.6;}}
  .controls{{padding:14px 40px;display:flex;gap:10px;align-items:center;border-bottom:1px solid var(--border);flex-wrap:wrap;}}
  .ctrl-label{{font-family:'IBM Plex Mono',monospace;font-size:.65rem;color:var(--muted);text-transform:uppercase;letter-spacing:.1em;margin-right:2px;}}
  .btn{{background:transparent;border:1px solid var(--border);color:var(--muted);padding:5px 13px;border-radius:3px;
    font-family:'IBM Plex Mono',monospace;font-size:.68rem;cursor:pointer;transition:all .15s;letter-spacing:.05em;}}
  .btn:hover{{border-color:var(--accent);color:var(--accent);}}
  .btn.active{{background:var(--accent);border-color:var(--accent);color:#000;font-weight:600;}}
  .main{{padding:28px 40px;display:flex;flex-direction:column;gap:28px;}}
  .section-label{{font-family:'IBM Plex Mono',monospace;font-size:.62rem;color:var(--accent);letter-spacing:.2em;text-transform:uppercase;margin-bottom:12px;}}
  .stats-row{{display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:14px;margin-bottom:28px;}}
  .stat-card{{background:var(--panel);border:1px solid var(--border);border-radius:5px;padding:14px 18px;}}
  .stat-value{{font-family:'Bebas Neue',sans-serif;font-size:1.75rem;color:#fff;letter-spacing:.05em;line-height:1;}}
  .stat-label{{font-family:'IBM Plex Mono',monospace;font-size:.6rem;color:var(--muted);text-transform:uppercase;letter-spacing:.12em;margin-top:4px;}}
  .card{{background:var(--panel);border:1px solid var(--border);border-radius:6px;padding:22px;position:relative;overflow:hidden;}}
  .card::before{{content:'';position:absolute;top:0;left:0;right:0;height:2px;background:linear-gradient(90deg,var(--accent),transparent);}}
  .card-title{{font-family:'Bebas Neue',sans-serif;font-size:1.2rem;letter-spacing:.08em;color:#fff;margin-bottom:2px;}}
  .card-sub{{font-size:.72rem;color:var(--muted);margin-bottom:16px;}}
  .legend{{display:flex;flex-wrap:wrap;gap:10px 20px;margin-bottom:14px;}}
  .legend-item{{display:flex;align-items:center;gap:6px;font-family:'IBM Plex Mono',monospace;font-size:.65rem;color:var(--text);cursor:pointer;transition:opacity .2s;}}
  .legend-item.hidden{{opacity:.3;}}
  .legend-dot{{width:10px;height:10px;border-radius:2px;flex-shrink:0;}}
  .total-wrap{{height:340px;position:relative;}}
  .facility-grid{{display:grid;grid-template-columns:repeat(auto-fill,minmax(400px,1fr));gap:18px;}}
  .facility-card{{background:var(--panel);border:1px solid var(--border);border-radius:6px;padding:18px;position:relative;overflow:hidden;}}
  .fac-name{{font-family:'Bebas Neue',sans-serif;font-size:1.1rem;letter-spacing:.08em;color:#fff;margin-bottom:2px;}}
  .fac-meta{{font-family:'IBM Plex Mono',monospace;font-size:.62rem;color:var(--muted);margin-bottom:12px;}}
  .fac-wrap{{height:170px;position:relative;}}
</style>
</head>
<body>
<header>
  <div class="logo-mark"></div>
  <div>
    <div class="subtitle">S&P Global PointLogic</div>
    <h1>LNG Feedgas Dashboard</h1>
  </div>
  <div class="date-badge">
    <div>{date_range}</div>
    <div class="updated">Updated {generated_at}</div>
  </div>
</header>
<div class="controls">
  <span class="ctrl-label">View:</span>
  <button class="btn active" id="btn-daily" onclick="setView('daily')">Daily</button>
  <button class="btn" id="btn-weekly" onclick="setView('weekly')">7-Day Avg</button>
  <button class="btn" id="btn-monthly" onclick="setView('monthly')">Monthly</button>
  <span class="ctrl-label" style="margin-left:16px">Range:</span>
  <button class="btn" id="btn-30" onclick="setRange(30)">30D</button>
  <button class="btn" id="btn-90" onclick="setRange(90)">90D</button>
  <button class="btn" id="btn-180" onclick="setRange(180)">180D</button>
  <button class="btn active" id="btn-0" onclick="setRange(0)">All</button>
</div>
<div class="main">
  <div>
    <div class="section-label">// Summary Statistics</div>
    <div class="stats-row" id="statsRow"></div>
  </div>
  <div>
    <div class="section-label">// Total US LNG Feedgas — All Facilities Stacked</div>
    <div class="card">
      <div class="card-title">All Facilities Combined</div>
      <div class="card-sub">MMcf/d · Click legend items to toggle facilities</div>
      <div class="legend" id="stackLegend"></div>
      <div class="total-wrap"><canvas id="totalChart"></canvas></div>
    </div>
  </div>
  <div>
    <div class="section-label">// Breakdown by Facility</div>
    <div class="facility-grid" id="facilityGrid"></div>
  </div>
</div>

<script>
const RAW = {data_str};
const FACILITIES = {facilities_js};
const COLORS = {colors_js};

let currentView = 'daily', currentRange = 0;
let hiddenFacilities = new Set();
let totalChartObj = null, facilityChartObjs = {{}};

function parseByDate(raw) {{
  const m = {{}};
  raw.forEach(r => {{ if(!m[r.d]) m[r.d]={{}}; m[r.d][r.f]=(m[r.d][r.f]||0)+r.v; }});
  return m;
}}

function filteredDates(byDate) {{
  const dates = Object.keys(byDate).sort();
  if(currentRange===0) return dates;
  const last = new Date(dates[dates.length-1]+'T00:00:00');
  const cutoff = new Date(last); cutoff.setDate(cutoff.getDate()-currentRange);
  return dates.filter(d => new Date(d+'T00:00:00') >= cutoff);
}}

function aggregate(dates, byDate, facility) {{
  const raw = dates.map(d => byDate[d]?.[facility] ?? null);
  if(currentView==='daily') return {{labels:dates, values:raw}};
  if(currentView==='weekly') {{
    const out=[],lbls=[];
    for(let i=6;i<dates.length;i++) {{
      const w=raw.slice(i-6,i+1).filter(v=>v!=null);
      out.push(w.length?w.reduce((a,b)=>a+b,0)/w.length:null); lbls.push(dates[i]);
    }}
    return {{labels:lbls,values:out}};
  }}
  const mon={{}};
  dates.forEach((d,i) => {{ const k=d.slice(0,7); if(!mon[k]) mon[k]=[]; if(raw[i]!=null) mon[k].push(raw[i]); }});
  const lbls=Object.keys(mon).sort();
  return {{labels:lbls, values:lbls.map(k=>mon[k].length?mon[k].reduce((a,b)=>a+b,0)/mon[k].length:null)}};
}}

function fmtLbl(d) {{
  if(currentView==='monthly') {{
    const [y,m]=d.split('-');
    return new Date(y,m-1).toLocaleString('default',{{month:'short',year:'2-digit'}});
  }}
  return new Date(d+'T00:00:00').toLocaleDateString('en-US',{{month:'short',day:'numeric'}});
}}

function hexToRgb(hex) {{
  const r=/^#?([a-f\\d]{{2}})([a-f\\d]{{2}})([a-f\\d]{{2}})$/i.exec(hex);
  return r?`${{parseInt(r[1],16)}},${{parseInt(r[2],16)}},${{parseInt(r[3],16)}}`:'0,180,216';
}}

const baseOpts = {{
  responsive:true, maintainAspectRatio:false,
  interaction:{{mode:'index',intersect:false}},
  plugins:{{
    legend:{{display:false}},
    tooltip:{{
      backgroundColor:'#0d1420',borderColor:'#1e2d45',borderWidth:1,
      titleFont:{{family:'IBM Plex Mono',size:10}},
      bodyFont:{{family:'IBM Plex Mono',size:10}},
      callbacks:{{label:ctx=>` ${{ctx.dataset.label}}: ${{ctx.parsed.y?.toFixed(0)||0}} MMcf/d`}}
    }}
  }},
  scales:{{
    x:{{grid:{{color:'rgba(255,255,255,0.04)',drawBorder:false}},ticks:{{color:'#64748b',font:{{family:'IBM Plex Mono',size:9}},maxTicksLimit:12}},border:{{display:false}}}},
    y:{{grid:{{color:'rgba(255,255,255,0.04)',drawBorder:false}},ticks:{{color:'#64748b',font:{{family:'IBM Plex Mono',size:9}},callback:v=>v>=1000?`${{(v/1000).toFixed(0)}}k`:v}},border:{{display:false}}}}
  }}
}};

function buildLegend() {{
  const leg=document.getElementById('stackLegend'); leg.innerHTML='';
  FACILITIES.forEach(f => {{
    const item=document.createElement('div');
    item.className='legend-item'+(hiddenFacilities.has(f)?' hidden':'');
    item.innerHTML=`<div class="legend-dot" style="background:${{COLORS[f]}}"></div>${{f}}`;
    item.onclick=()=>{{ hiddenFacilities.has(f)?hiddenFacilities.delete(f):hiddenFacilities.add(f); render(); }};
    leg.appendChild(item);
  }});
}}

function buildTotalChart(byDate) {{
  const dates=filteredDates(byDate);
  const {{labels}}=aggregate(dates,byDate,FACILITIES[0]);
  const datasets=FACILITIES.filter(f=>!hiddenFacilities.has(f)).map(f=>{{
    const {{values}}=aggregate(dates,byDate,f);
    return {{label:f,data:values,backgroundColor:COLORS[f],borderColor:'transparent',borderWidth:0,barPercentage:1.0,categoryPercentage:0.92}};
  }});
  if(totalChartObj) totalChartObj.destroy();
  const ctx=document.getElementById('totalChart').getContext('2d');
  totalChartObj=new Chart(ctx,{{
    type:'bar', data:{{labels:labels.map(fmtLbl),datasets}},
    options:{{...baseOpts,
      scales:{{
        x:{{...baseOpts.scales.x,stacked:true}},
        y:{{...baseOpts.scales.y,stacked:true}}
      }},
      plugins:{{...baseOpts.plugins,tooltip:{{...baseOpts.plugins.tooltip,
        callbacks:{{
          label:ctx=>` ${{ctx.dataset.label}}: ${{ctx.parsed.y?.toFixed(0)||0}} MMcf/d`,
          footer:items=>`Total: ${{items.reduce((s,i)=>s+(i.parsed.y||0),0).toFixed(0)}} MMcf/d`
        }}
      }}}}
    }}
  }});
}}

function buildFacilityCharts(byDate) {{
  const grid=document.getElementById('facilityGrid'); grid.innerHTML='';
  const dates=filteredDates(byDate);
  FACILITIES.forEach(f=>{{
    const {{labels,values}}=aggregate(dates,byDate,f);
    const valid=values.filter(v=>v!=null&&v>0);
    if(!valid.length) return;
    const avg=valid.reduce((a,b)=>a+b,0)/valid.length;
    const peak=Math.max(...valid);
    const latest=valid[valid.length-1];
    const color=COLORS[f];
    const card=document.createElement('div');
    card.className='facility-card';
    card.innerHTML=`
      <div style="position:absolute;top:0;left:0;right:0;height:2px;background:${{color}}"></div>
      <div class="fac-name">${{f}}</div>
      <div class="fac-meta">Avg ${{avg.toFixed(0)}} &nbsp;·&nbsp; Peak ${{peak.toFixed(0)}} &nbsp;·&nbsp; Latest ${{latest?.toFixed(0)||'—'}} <span style="color:var(--muted)">MMcf/d</span></div>
      <div class="fac-wrap"><canvas id="fc_${{f.replace(/[^a-z]/gi,'_')}}"></canvas></div>`;
    grid.appendChild(card);
    const ctx2=document.getElementById(`fc_${{f.replace(/[^a-z]/gi,'_')}}`).getContext('2d');
    const grad=ctx2.createLinearGradient(0,0,0,150);
    const rgb=hexToRgb(color);
    grad.addColorStop(0,`rgba(${{rgb}},.22)`); grad.addColorStop(1,`rgba(${{rgb}},0)`);
    if(facilityChartObjs[f]) facilityChartObjs[f].destroy();
    facilityChartObjs[f]=new Chart(ctx2,{{
      type:'line',
      data:{{labels:labels.map(fmtLbl),datasets:[{{data:values,borderColor:color,backgroundColor:grad,borderWidth:1.5,fill:true,tension:0.3,pointRadius:0,pointHoverRadius:3,spanGaps:true}}]}},
      options:{{...baseOpts,scales:{{x:{{...baseOpts.scales.x}},y:{{...baseOpts.scales.y,stacked:false}}}},
        plugins:{{...baseOpts.plugins,tooltip:{{...baseOpts.plugins.tooltip,callbacks:{{label:ctx=>` ${{ctx.parsed.y?.toFixed(0)||0}} MMcf/d`}}}}}}
      }}
    }});
  }});
}}

function buildStats(byDate) {{
  const dates=filteredDates(byDate);
  const totals=dates.map(d=>FACILITIES.reduce((s,f)=>s+(byDate[d]?.[f]||0),0));
  const latest=totals[totals.length-1]||0;
  const avg=totals.reduce((a,b)=>a+b,0)/totals.length;
  const peak=Math.max(...totals);
  const active=FACILITIES.filter(f=>dates.some(d=>byDate[d]?.[f]>0)).length;
  document.getElementById('statsRow').innerHTML=`
    <div class="stat-card"><div class="stat-value">${{latest.toFixed(0)}}</div><div class="stat-label">Latest MMcf/d</div></div>
    <div class="stat-card"><div class="stat-value">${{avg.toFixed(0)}}</div><div class="stat-label">Period Avg MMcf/d</div></div>
    <div class="stat-card"><div class="stat-value">${{peak.toFixed(0)}}</div><div class="stat-label">Peak Day MMcf/d</div></div>
    <div class="stat-card"><div class="stat-value">${{active}}</div><div class="stat-label">Active Facilities</div></div>
    <div class="stat-card"><div class="stat-value">${{dates.length}}</div><div class="stat-label">Days of Data</div></div>`;
}}

function render() {{
  const byDate=parseByDate(RAW);
  buildStats(byDate); buildLegend(); buildTotalChart(byDate); buildFacilityCharts(byDate);
}}

function setView(v) {{
  currentView=v;
  ['daily','weekly','monthly'].forEach(x=>document.getElementById('btn-'+x)?.classList.toggle('active',x===v));
  render();
}}

function setRange(r) {{
  currentRange=r;
  [0,30,90,180].forEach(x=>document.getElementById('btn-'+x)?.classList.toggle('active',x===r));
  render();
}}

render();
</script>
</body>
</html>"""

    with open(HTML_FILE, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"   ✓ Dashboard saved to {HTML_FILE}")


def check_git_installed():
    """Check if Git is installed and accessible."""
    try:
        result = subprocess.run(["git", "--version"], capture_output=True, text=True, check=True)
        return True
    except (FileNotFoundError, subprocess.CalledProcessError):
        return False


def git_commit_and_push():
    """Commit changes and push to GitHub."""
    print("\n🔄 Pushing to GitHub...")
    
    # Check if Git is installed
    if not check_git_installed():
        print("\n   ⚠️  Git is not installed or not in PATH")
        print("   📥 Download Git from: https://git-scm.com/download/win")
        print("   After installing, restart your terminal and try again.")
        print("\n   ℹ️  Dashboard files have been updated locally at:")
        print(f"   {HTML_FILE}")
        return False
    
    os.chdir(REPO_DIR)
    
    # Check if this is a git repository
    git_check = subprocess.run(["git", "rev-parse", "--git-dir"], capture_output=True, text=True)
    if git_check.returncode != 0:
        print("\n   ⚠️  This directory is not a Git repository")
        print("   Run these commands to set up Git:")
        print(f"\n   cd {REPO_DIR}")
        print("   git init")
        print("   git remote add origin https://github.com/YOUR-USERNAME/LNGFeedgas.git")
        print("   git add .")
        print('   git commit -m "Initial commit"')
        print("   git branch -M main")
        print("   git push -u origin main")
        return False
    
    # Configure git if needed (only needs to run once, but harmless to repeat)
    subprocess.run(["git", "config", "user.name", "LNG Dashboard Bot"], check=False)
    subprocess.run(["git", "config", "user.email", "melaea@example.com"], check=False)
    
    # Stage all changes
    subprocess.run(["git", "add", "."], capture_output=True, text=True)
    
    # Check if there are changes to commit
    status = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True)
    
    if not status.stdout.strip():
        print("   ℹ️  No changes to commit")
        return True
    
    # Commit with timestamp
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    commit_msg = f"Auto-update dashboard - {timestamp}"
    
    try:
        subprocess.run(["git", "commit", "-m", commit_msg], check=True, capture_output=True, text=True)
        print(f"   ✓ Changes committed: {commit_msg}")
    except subprocess.CalledProcessError as e:
        print(f"   ⚠️  Commit failed: {e.stderr}")
        return False
    
    # Push to GitHub
    try:
        subprocess.run(["git", "push"], capture_output=True, text=True, check=True)
        print(f"   ✓ Successfully pushed to GitHub")
        return True
    except subprocess.CalledProcessError as e:
        print(f"   ⚠️  Push failed: {e.stderr}")
        print("\n   You may need to:")
        print("   1. Set up authentication (GitHub Personal Access Token or SSH)")
        print("   2. Run: git push -u origin main")
        return False


def main():
    print("=" * 60)
    print("  LNG Feedgas Dashboard — Auto-Update & Deploy")
    print(f"  {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 60)

    try:
        # Step 1: Fetch data
        print("\n[1/4] Fetching data from S&P Global API...")
        feedgas = fetch_data()

        # Step 2: Save data
        print("\n[2/4] Saving data...")
        save_data(feedgas)

        # Step 3: Generate HTML
        print("\n[3/4] Generating HTML dashboard...")
        generate_html(feedgas)

        # Step 4: Push to GitHub
        print("\n[4/4] Deploying to GitHub...")
        git_success = git_commit_and_push()

        print("\n" + "=" * 60)
        if git_success:
            print("  ✅ Dashboard updated and deployed successfully!")
            print(f"  📂 Local: {HTML_FILE}")
            print(f"  🌐 GitHub: Check your repository")
        else:
            print("  ⚠️  Dashboard updated locally (GitHub push skipped)")
            print(f"  📂 Local: {HTML_FILE}")
            print("  💡 See instructions above to complete Git setup")
        print("=" * 60)

    except requests.exceptions.RequestException as e:
        print(f"\n❌ API error: {e}")
    except Exception as e:
        print(f"\n❌ Unexpected error: {e}")
        raise


if __name__ == "__main__":
    main()


# ══════════════════════════════════════════════════════════════════════════════
# FIRST-TIME SETUP
# ══════════════════════════════════════════════════════════════════════════════
#
# 1. Make sure Git is installed:
#      - Download from: https://git-scm.com/download/win
#      - Or check: run "git --version" in Command Prompt
#
# 2. Initialize your GitHub repo (if not already done):
#      cd C:/Users/melaea/LNGFeedgas
#      git init
#      git remote add origin https://github.com/YOUR-USERNAME/LNGFeedgas.git
#
# 3. First commit:
#      git add .
#      git commit -m "Initial commit"
#      git branch -M main
#      git push -u origin main
#
# 4. Enable GitHub Pages (for live URL):
#      - Go to your repo on GitHub
#      - Settings → Pages → Source: Deploy from branch → Branch: main → Save
#      - Your dashboard will be live at: https://YOUR-USERNAME.github.io/LNGFeedgas/
#
# ══════════════════════════════════════════════════════════════════════════════
# AUTO-RUN DAILY (Windows Task Scheduler)
# ══════════════════════════════════════════════════════════════════════════════
#
# 1. Find your Python path:
#       where python
#    Example: C:/Users/melaea/AppData/Local/Programs/Python/Python311/python.exe
#
# 2. Open Task Scheduler (search in Start Menu)
#
# 3. Create Basic Task:
#    - Name: LNG Dashboard Auto-Update
#    - Trigger: Daily → 7:00 AM (or whenever you want)
#    - Action: Start a program
#      - Program:  C:/Users/melaea/AppData/Local/Programs/Python/Python311/python.exe
#      - Arguments: update_and_deploy.py
#      - Start in:  C:/Users/melaea/LNGFeedgas
#
# 4. Click Finish — it will update daily automatically!
#
# ══════════════════════════════════════════════════════════════════════════════

  LNG Feedgas Dashboard — Auto-Update & Deploy
  2026-03-16 10:28:39

[1/4] Fetching data from S&P Global API...
📡 Fetching data from 1/1/2025 to 3/16/2026...
   ✓ 3779 feedgas records fetched

[2/4] Saving data...
   ✓ Data saved to C:\Users\melaea\LNGFeedgas\lng_feedgas_clean.json

[3/4] Generating HTML dashboard...
   ✓ Dashboard saved to C:\Users\melaea\LNGFeedgas\index.html

[4/4] Deploying to GitHub...

🔄 Pushing to GitHub...

   ⚠️  Git is not installed or not in PATH
   📥 Download Git from: https://git-scm.com/download/win
   After installing, restart your terminal and try again.

   ℹ️  Dashboard files have been updated locally at:
   C:\Users\melaea\LNGFeedgas\index.html

  ⚠️  Dashboard updated locally (GitHub push skipped)
  📂 Local: C:\Users\melaea\LNGFeedgas\index.html
  💡 See instructions above to complete Git setup
